In [5]:
! pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 27.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.7 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 93.6 MB/s  0:00:00
  Attempting uninstall: fsspec0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/14 [propcache]
    Found existing installation: fsspec 2026.3.0━━━━━━━━━━━━━━━━━━  4/14 [fsspec]
    Uninstalling fsspec-2026.3.0:m━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/14 [fsspec]
      Successfully uninstalled fsspec-2026.3.0━━━━━━━━━━━━━━━━  4/14 [fsspec]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [datasets]/14 [datasets]ess]lls]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import polars as pl
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import torch
from torch.nn import CrossEntropyLoss

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from pathlib import Path
import json
import math
import pyarrow.parquet as pq

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DATA_PATH = "./data/cicids2017_after_EDA.parquet"
LABEL_COL = "Label"
TEXT_COL = "Text"

MODEL_NAME = "markusbayer/CySecBERT"  # or your CysecBERT base
MAX_LENGTH = 512
BATCH_SIZE = 10_000  # safe for 16GB RAM

## Phase 1 - Train test validation set split

In [6]:
lf = pl.scan_parquet(DATA_PATH)

In [7]:
out_dir = Path("splits_tmp")
out_dir.mkdir(exist_ok=True)

train_parts = []
val_parts = []
test_parts = []

In [8]:
labels = (
    lf.select(LABEL_COL)
      .unique()
      .collect()[LABEL_COL]
      .to_list()
)

for label in labels:
    # Load only ONE label into memory
    df_label = (
        lf.filter(pl.col(LABEL_COL) == label)
          .collect(streaming=True)
    )

    n = df_label.height
    idx = np.random.permutation(n)

    train_end = int(0.8 * n)
    val_end = int(0.9 * n)

    train_df = df_label[idx[:train_end]]
    val_df   = df_label[idx[train_end:val_end]]
    test_df  = df_label[idx[val_end:]]

    # Write immediately (don’t keep in RAM)
    train_path = out_dir / f"train_{label}.parquet"
    val_path   = out_dir / f"val_{label}.parquet"
    test_path  = out_dir / f"test_{label}.parquet"

    train_df.write_parquet(train_path)
    val_df.write_parquet(val_path)
    test_df.write_parquet(test_path)

    train_parts.append(train_path)
    val_parts.append(val_path)
    test_parts.append(test_path)

    # Explicitly free memory
    del df_label, train_df, val_df, test_df

/tmp/ipykernel_24085/531970717.py:12: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


In [9]:
train_df = pl.concat([pl.read_parquet(p) for p in train_parts])
val_df   = pl.concat([pl.read_parquet(p) for p in val_parts])
test_df  = pl.concat([pl.read_parquet(p) for p in test_parts])

In [10]:
train_df.write_parquet("train_split.parquet")
val_df.write_parquet("val_split.parquet")
test_df.write_parquet("test_split.parquet")


In [12]:
print(train_df.shape, val_df.shape, test_df.shape)

(2248760, 74) (281095, 74) (281103, 74)


## Phase 2 - Calculating class weights (train only)

In [17]:
train_df = pl.read_parquet("train_split.parquet")

In [19]:
class_counts = (
    train_df
    .select(LABEL_COL)
    .to_series()
    .value_counts()
    .sort("count", descending=True)
)

print(class_counts)


shape: (15, 2)
┌────────────────────────────┬─────────┐
│ Label                      ┆ count   │
│ ---                        ┆ ---     │
│ str                        ┆ u32     │
╞════════════════════════════╪═════════╡
│ BENIGN                     ┆ 1809592 │
│ DoS Hulk                   ┆ 178050  │
│ PortScan                   ┆ 127043  │
│ DDoS                       ┆ 102420  │
│ DoS GoldenEye              ┆ 8234    │
│ …                          ┆ …       │
│ Web Attack – Brute Force   ┆ 1205    │
│ Web Attack – XSS           ┆ 521     │
│ Infiltration               ┆ 28      │
│ Web Attack – Sql Injection ┆ 16      │
│ Heartbleed                 ┆ 8       │
└────────────────────────────┴─────────┘


In [20]:
total_samples = train_df.height
num_classes = class_counts.height

class_weights = {}

for row in class_counts.iter_rows(named=True):
    label = row[LABEL_COL]
    count = row["count"]
    weight = total_samples / (num_classes * count)
    class_weights[label] = float(weight)

In [21]:
for k, v in sorted(class_weights.items(), key=lambda x: x[1], reverse=True):
    print(f"{k:25s} -> {v:.4f}")


Heartbleed                -> 18739.6667
Web Attack – Sql Injection -> 9369.8333
Infiltration              -> 5354.1905
Web Attack – XSS          -> 287.7492
Web Attack – Brute Force  -> 124.4127
Bot                       -> 95.8551
DoS Slowhttptest          -> 34.0799
DoS slowloris             -> 32.4848
SSH-Patator               -> 31.7823
FTP-Patator               -> 23.6165
DoS GoldenEye             -> 18.2071
DDoS                      -> 1.4638
PortScan                  -> 1.1801
DoS Hulk                  -> 0.8420
BENIGN                    -> 0.0828


In [22]:
out_dir = Path("data")
out_dir.mkdir(exist_ok=True)

weights_path = out_dir / "class_weights.json"

with open(weights_path, "w") as f:
    json.dump(class_weights, f, indent=2)

print("Saved class weights to:", weights_path)


Saved class weights to: data/class_weights.json


In [24]:
log_class_weights = {}

for row in class_counts.iter_rows(named=True):
    label = row[LABEL_COL]
    count = row["count"]
    
    # Original inverse frequency
    original_weight = total_samples / (num_classes * count)
    
    # Log-scaled smoothing (log(1 + x) prevents weights from dropping to 0 or below)
    smoothed_weight = math.log(1 + original_weight)
    
    log_class_weights[label] = float(smoothed_weight)

# Print the new distributed weights
print("Log-scaled smoothed weights:")
for k, v in sorted(log_class_weights.items(), key=lambda x: x[1], reverse=True):
    print(f"{k:25s} -> {v:.4f}")

# Save to a new JSON file
out_dir = Path("data")
out_dir.mkdir(exist_ok=True)
log_weights_path = out_dir / "class_weights_log_scaled.json"

with open(log_weights_path, "w") as f:
    json.dump(log_class_weights, f, indent=2)

print(f"\nSaved log-scaled class weights to: {log_weights_path}")

Log-scaled smoothed weights:
Heartbleed                -> 9.8385
Web Attack – Sql Injection -> 9.1454
Infiltration              -> 8.5858
Web Attack – XSS          -> 5.6656
Web Attack – Brute Force  -> 4.8316
Bot                       -> 4.5732
DoS Slowhttptest          -> 3.5576
DoS slowloris             -> 3.5111
SSH-Patator               -> 3.4899
FTP-Patator               -> 3.2034
DoS GoldenEye             -> 2.9553
DDoS                      -> 0.9017
PortScan                  -> 0.7793
DoS Hulk                  -> 0.6108
BENIGN                    -> 0.0796

Saved log-scaled class weights to: data/class_weights_log_scaled.json


## Phase 3 - Tokenization and hugging face dataset

In [8]:
data_files = {
    "train": "train_split.parquet",
    "validation": "val_split.parquet",
    "test": "test_split.parquet",
}

raw_datasets = load_dataset(
    "parquet",
    data_files=data_files
)


Generating train split: 2248760 examples [00:03, 747955.75 examples/s]
Generating validation split: 281095 examples [00:11, 24552.30 examples/s]
Generating test split: 281103 examples [00:03, 78619.32 examples/s]


In [4]:

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)



In [5]:

def tokenize_fn(batch):
    feature_cols = [
        k for k in batch.keys()
        if k not in [LABEL_COL, "Timestamp"]
    ]

    texts = [
        " ".join(str(batch[col][i]) for col in feature_cols)
        for i in range(len(batch[LABEL_COL]))
    ]

    enc = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

    enc["label"] = batch[LABEL_COL]
    return enc


In [9]:
tokenized_datasets = {}

for split in ["train", "validation", "test"]:
    tokenized_datasets[split] = raw_datasets[split].map(
        tokenize_fn,
        batched=True,
        batch_size=512,   # SAFE
        remove_columns=raw_datasets[split].column_names,
        num_proc=4
    )


Map (num_proc=4): 100%|██████████| 281103/281103 [01:05<00:00, 4312.68 examples/s]


In [12]:
tokenized_datasets["train"].save_to_disk("data/train_dataset")
tokenized_datasets["validation"].save_to_disk("data/val_dataset")
tokenized_datasets["test"].save_to_disk("data/test_dataset")

Saving the dataset (1/1 shards): 100%|██████████| 281103/281103 [00:01<00:00, 196097.23 examples/s]


In [13]:
sample = next(iter(tokenized_datasets["train"]))
len(sample["input_ids"])

280

In [14]:
print(tokenized_datasets)

{'train': Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'label'],
    num_rows: 2248760
}), 'validation': Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'label'],
    num_rows: 281095
}), 'test': Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'label'],
    num_rows: 281103
})}
